# Nepali TTS Dataset Builder: YouTube → Denoised Chunks → ASR → ZIP

This notebook is designed for a distributed dataset collection team. Each person gives one YouTube video link, and the notebook produces a clean, TTS-ready dataset package.

Pipeline:
1. Download audio from YouTube with `yt-dlp`.
2. Denoise using FullSubNet+.
3. Split clean speech using Silero VAD into roughly 4-8 second chunks.
4. Transcribe with the Nepali IndicConformer `.nemo` ASR model.
5. Validate audio/transcription coverage.
6. Export a ZIP containing `16000Hz`, `22050Hz`, `dataset_card.md`, and audit CSVs.

Speaker filtering is intentionally disabled for now, as requested.

## 0. Runtime

In Colab, choose:

`Runtime` → `Change runtime type` → `T4 GPU` or better.

The Nepali ASR model download is about 498 MiB. FullSubNet+ checkpoint download is about 100 MiB.

In [ ]:
import os
import re
import sys
import json
import time
import math
import shutil
import zipfile
import subprocess
from pathlib import Path

import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Fill These Inputs

Use one clean Nepali speech/audiobook video per person when possible. If the video is longer than one hour, the notebook can cap the exported dataset to about one hour.

In [ ]:
# @title Dataset Input
YOUTUBE_URL = "https://www.youtube.com/watch?v=6wesUbJbn78"  # @param {type:"string"}
DATASET_ID = "nepali_voice_001"  # @param {type:"string"}
LANGUAGE_ID = "ne"  # @param {type:"string"}

# If False, a playlist URL downloads only the first/current video.
ALLOW_PLAYLIST = False  # @param {type:"boolean"}

# Keep the final dataset near one hour. Disable if you want all chunks from the video.
TRIM_TO_TARGET_HOUR = True  # @param {type:"boolean"}
TARGET_SECONDS = 3600  # @param {type:"integer"}
MIN_EXPECTED_SECONDS = 3000  # @param {type:"integer"}

# ASR mode. Default is the Nepali .nemo model. The fallback is slower and less controlled.
ASR_ENGINE = "nemo_nepali_nemo"  # @param ["nemo_nepali_nemo", "ai4bharat_transformers_fallback"]
ASR_BATCH_SIZE = 8  # @param {type:"integer"}

# Denoising settings.
FULLSUBNET_BATCH_SIZE = 8  # @param {type:"integer"}
USE_FP16_DENOISER = True  # @param {type:"boolean"}

assert YOUTUBE_URL.strip(), "Please paste a YouTube URL before running the notebook."

def slugify(value: str, fallback: str = "dataset") -> str:
    value = re.sub(r"[^A-Za-z0-9_\-]+", "_", value.strip())
    value = re.sub(r"_+", "_", value).strip("_")
    return value or fallback

DATASET_ID = slugify(DATASET_ID, "nepali_voice_001")
WORK_DIR = Path("/content/nepali_tts_dataset_builder")
RAW_DIR = WORK_DIR / "raw_youtube_audio"
MODEL_DIR = WORK_DIR / "models"
TMP_PROCESSED_ROOT = WORK_DIR / "tmp_processed_audio"
FINAL_ROOT = WORK_DIR / "final_dataset"
FINAL_16K = FINAL_ROOT / "16000Hz" / DATASET_ID
FINAL_22K = FINAL_ROOT / "22050Hz" / DATASET_ID

for path in [RAW_DIR, MODEL_DIR, TMP_PROCESSED_ROOT, FINAL_16K, FINAL_22K]:
    path.mkdir(parents=True, exist_ok=True)

print("Dataset ID:", DATASET_ID)
print("Work dir:", WORK_DIR)

## 2. Install Dependencies

This can take several minutes the first time. If Colab asks to restart after dependency installation, restart the runtime and run cells again from the top.

In [ ]:
def run(cmd, *, cwd=None, check=True):
    print("\n$", " ".join(map(str, cmd)) if isinstance(cmd, (list, tuple)) else cmd)
    return subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), check=check)

run("apt-get -qq update", check=True)
run("apt-get -qq install -y ffmpeg aria2 sox libsndfile1", check=True)

base_packages = [
    "yt-dlp",
    "gdown",
    "librosa>=0.10",
    "soundfile>=0.12",
    "tqdm>=4.66",
    "pandas",
    "numpy",
    "scipy",
    "matplotlib",
    "ipython",
    "cython",
    "packaging",
    "huggingface_hub",
]

asr_packages = []
if ASR_ENGINE == "nemo_nepali_nemo":
    run("apt-get install -y python3.10 python3.10-venv python3.10-dev", check=True)
    run("python3.10 -m venv /opt/nemo_env", check=True)
    run("/opt/nemo_env/bin/python -m pip install -U pip", check=True)
    run(f"/opt/nemo_env/bin/python -m pip install -q {' '.join(base_packages)}", check=True)
    run("/opt/nemo_env/bin/python -m pip install -q nemo_toolkit[asr]", check=True)
else:
    asr_packages.extend([
        "transformers",
        "accelerate",
        "sentencepiece",
        "onnxruntime-gpu; platform_machine == 'x86_64'",
    ])

run([sys.executable, "-m", "pip", "install", "-U", "-q", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", *base_packages, *asr_packages])

print("Dependency install finished.")

## 3. Unpack FullSubNet+ Preprocessing Code

The preprocessing code is embedded in this notebook so the team does not need access to Samir's server paths.

In [ ]:
import base64
import io

FULLSUBNET_CODE_ZIP_B64 = """..."""

code_zip = base64.b64decode(FULLSUBNET_CODE_ZIP_B64)
with zipfile.ZipFile(io.BytesIO(code_zip), "r") as archive:
    archive.extractall(WORK_DIR)

DENOISER_DIR = WORK_DIR / "fullsubnet_plus_denoiser"
print("FullSubNet+ code ready:", DENOISER_DIR)
print("Files:", sorted(p.name for p in DENOISER_DIR.glob("*.py")))

## 4. Download Models

This downloads:
- FullSubNet+ denoiser checkpoint.
- Nepali IndicConformer `.nemo` ASR model.

In [ ]:
import gdown

FULLSUBNET_GDRIVE_ID = "1UJSt1G0P_aXry-u79LLU_l9tCnNa2u7C"
FULLSUBNET_CHECKPOINT = MODEL_DIR / "fullsubnet_best_model.tar"

NEMO_URL = "https://objectstore.e2enetworks.net/indicconformer/models/indicconformer_stt_ne_hybrid_rnnt_large.nemo"
NEMO_MODEL_PATH = MODEL_DIR / "indicconformer_stt_ne_hybrid_rnnt_large.nemo"

if not FULLSUBNET_CHECKPOINT.exists():
    print("Downloading FullSubNet+ checkpoint...")
    gdown.download(id=FULLSUBNET_GDRIVE_ID, output=str(FULLSUBNET_CHECKPOINT), quiet=False)

if ASR_ENGINE == "nemo_nepali_nemo" and not NEMO_MODEL_PATH.exists():
    print("Downloading Nepali IndicConformer .nemo model...")
    run([
        "aria2c",
        "-x", "8",
        "-s", "8",
        "-k", "1M",
        "-c",
        "-o", NEMO_MODEL_PATH.name,
        "-d", str(MODEL_DIR),
        NEMO_URL,
    ])

assert FULLSUBNET_CHECKPOINT.exists(), f"Missing FullSubNet checkpoint: {FULLSUBNET_CHECKPOINT}"
if ASR_ENGINE == "nemo_nepali_nemo":
    assert NEMO_MODEL_PATH.exists(), f"Missing NeMo ASR model: {NEMO_MODEL_PATH}"

print("FullSubNet checkpoint:", FULLSUBNET_CHECKPOINT, FULLSUBNET_CHECKPOINT.stat().st_size / 1024**2, "MiB")
if ASR_ENGINE == "nemo_nepali_nemo":
    print("NeMo ASR model:", NEMO_MODEL_PATH, NEMO_MODEL_PATH.stat().st_size / 1024**2, "MiB")

## 5. Download YouTube Audio

The audio is converted to mono 16 kHz WAV before denoising.

In [ ]:
shutil.rmtree(RAW_DIR, ignore_errors=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

output_template = (
    str(RAW_DIR / "%(playlist_index)03d_%(title).120B.%(ext)s")
    if ALLOW_PLAYLIST
    else str(RAW_DIR / "source.%(ext)s")
)

cmd = [
    "yt-dlp",
    "-f", "bestaudio/best",
    "--extract-audio",
    "--audio-format", "wav",
    "--audio-quality", "0",
    "--postprocessor-args", "ffmpeg:-ac 1 -ar 16000",
    "-o", output_template,
]
if not ALLOW_PLAYLIST:
    cmd.append("--no-playlist")
cmd.append(YOUTUBE_URL)

run(cmd)

raw_wavs = sorted(RAW_DIR.glob("*.wav"))
assert raw_wavs, f"No WAV was downloaded in {RAW_DIR}"
print("Downloaded WAV files:")
for wav in raw_wavs:
    print(" -", wav.name, f"{wav.stat().st_size / 1024**2:.1f} MiB")

## 6. Denoise and Split Audio

Output will be created in temporary `16000Hz` and `22050Hz` folders. Speaker filtering is skipped.

In [ ]:
shutil.rmtree(TMP_PROCESSED_ROOT, ignore_errors=True)
TMP_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

denoiser_device = "cuda" if torch.cuda.is_available() else "cpu"
preprocess_cmd = [
    sys.executable,
    str(DENOISER_DIR / "preprocess_youtube_dataset.py"),
    "--input_dir", str(RAW_DIR),
    "--output_root", str(TMP_PROCESSED_ROOT),
    "--checkpoint", str(FULLSUBNET_CHECKPOINT),
    "--device", denoiser_device,
    "--vad_device", "cpu",
    "--batch_size", str(FULLSUBNET_BATCH_SIZE),
    "--enhancer_chunk_size", "8.0",
    "--force",
]
if USE_FP16_DENOISER and denoiser_device == "cuda":
    preprocess_cmd.append("--fp16")

run(preprocess_cmd, cwd=DENOISER_DIR)

tmp_16k = TMP_PROCESSED_ROOT / "16000Hz"
tmp_22k = TMP_PROCESSED_ROOT / "22050Hz"
chunks_16k = sorted(tmp_16k.glob("*.wav"))
chunks_22k = sorted(tmp_22k.glob("*.wav"))
print("16 kHz chunks:", len(chunks_16k))
print("22.05 kHz chunks:", len(chunks_22k))
assert chunks_16k and chunks_22k, "Preprocessing produced no chunks."

## 7. Build the Final One-Hour Dataset Folder

If `TRIM_TO_TARGET_HOUR=True`, chunks are copied in order until the target duration is reached.

In [ ]:
import soundfile as sf
import pandas as pd

shutil.rmtree(FINAL_ROOT, ignore_errors=True)
FINAL_16K.mkdir(parents=True, exist_ok=True)
FINAL_22K.mkdir(parents=True, exist_ok=True)

def duration_seconds(path: Path) -> float:
    return float(sf.info(str(path)).duration)

selected = []
total_seconds = 0.0
for wav16 in sorted(tmp_16k.glob("*.wav")):
    wav22 = tmp_22k / wav16.name
    if not wav22.exists():
        print("Missing 22k pair, skipping:", wav16.name)
        continue
    dur = duration_seconds(wav16)
    if TRIM_TO_TARGET_HOUR and selected and total_seconds >= TARGET_SECONDS:
        break
    selected.append((wav16, wav22, dur))
    total_seconds += dur

assert selected, "No chunks selected for final dataset."

for wav16, wav22, _dur in selected:
    shutil.copy2(wav16, FINAL_16K / wav16.name)
    shutil.copy2(wav22, FINAL_22K / wav22.name)

print("Selected chunks:", len(selected))
print("Selected duration:", f"{total_seconds/60:.2f} minutes", f"({total_seconds/3600:.2f} hours)")
if total_seconds < MIN_EXPECTED_SECONDS:
    print("WARNING: Dataset is shorter than expected. Use a longer/cleaner YouTube source if you need exactly ~1 hour.")

## 8. Transcribe Chunks

Default ASR is the Nepali IndicConformer `.nemo` model. Transcription is resumable: if the CSV already exists, completed filenames are skipped.

In [ ]:
import csv
from tqdm.auto import tqdm

TRANSCRIPT_CSV = FINAL_16K / "transcriptions.csv"

def read_existing_transcripts(csv_path: Path) -> dict[str, str]:
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return {}
    rows = {}
    with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
        sample = handle.read(2048)
        handle.seek(0)
        delimiter = "|" if sample.splitlines() and "|" in sample.splitlines()[0] and "," not in sample.splitlines()[0] else ","
        reader = csv.DictReader(handle, delimiter=delimiter)
        for row in reader:
            name = (row.get("name") or row.get("wav") or row.get("audio") or "").strip()
            text = (row.get("transcription") or row.get("text") or "").strip()
            if name and text:
                rows[Path(name).name] = text
    return rows

def ensure_transcript_header(csv_path: Path):
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        with csv_path.open("w", encoding="utf-8", newline="") as handle:
            csv.writer(handle).writerow(["name", "transcription"])

def normalize_asr_item(item) -> str:
    if isinstance(item, str):
        return item.strip()
    if hasattr(item, "text"):
        return str(item.text).strip()
    if isinstance(item, (list, tuple)) and item:
        return normalize_asr_item(item[0])
    return str(item).strip()

def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]

ensure_transcript_header(TRANSCRIPT_CSV)
existing = read_existing_transcripts(TRANSCRIPT_CSV)
wavs = sorted(FINAL_16K.glob("*.wav"))
pending = [wav for wav in wavs if wav.name not in existing]

print("Total chunks:", len(wavs))
print("Already transcribed:", len(existing))
print("Pending:", len(pending))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if pending and ASR_ENGINE == "nemo_nepali_nemo":
    import tarfile
    import yaml
    import shutil
    import os
    import textwrap

    nemo_model_path_base = str(MODEL_DIR / "indicconformer_stt_ne_hybrid_rnnt_large.nemo")
    fixed_nemo_path = nemo_model_path_base.replace(".nemo", "_patched_v12.nemo")

    if not Path(fixed_nemo_path).exists():
        print(f"Patching {nemo_model_path_base} to restructure tokenizer files...")
        extract_dir = "tmp_nemo_extract"
        os.makedirs(extract_dir, exist_ok=True)
        with tarfile.open(nemo_model_path_base, "r") as tar:
            tar.extractall(extract_dir)
            
        config_path = os.path.join(extract_dir, "model_config.yaml")
        
        # Step 1: Clean duplicate keys in model_config.yaml
        with open(config_path, "r") as f:
            lines = f.readlines()
        clean_lines = []
        seen_keys = set()
        skip_block = False
        for line in lines:
            if not line.startswith(" ") and ":" in line:
                key = line.split(":")[0].strip()
                if key in seen_keys:
                    skip_block = True
                else:
                    seen_keys.add(key)
                    skip_block = False
            if not skip_block:
                clean_lines.append(line)
                
        with open(config_path, "w") as f:
            f.writelines(clean_lines)
            
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)

        # Step 2: In newer NeMo, we just need to provide 'dir' and 'type', but we MUST keep the original 'model_path'
        # exactly the same, so that NeMo's register_artifact matches the extracted tar exactly as before.
        # We don't move the tokenizer.model around anymore, because register_artifact wants it exactly where it was.
        def get_and_fix_tokenizer(d):
            if isinstance(d, dict):
                for k, v in list(d.items()):
                    if k == "tokenizer":
                        if not isinstance(v, dict):
                            d[k] = {}
                        d[k]["dir"] = "."
                        d[k]["type"] = "bpe"
                        
                        # Strip absolute paths if any exist (e.g. if the original config included absolute paths)
                        if "model_path" in d[k] and d[k]["model_path"]:
                            d[k]["model_path"] = os.path.basename(d[k]["model_path"])
                        if "vocab_path" in d[k] and d[k]["vocab_path"]:
                            d[k]["vocab_path"] = os.path.basename(d[k]["vocab_path"])
                    else:
                        get_and_fix_tokenizer(v)
            elif isinstance(d, list):
                for item in d:
                    get_and_fix_tokenizer(item)

        get_and_fix_tokenizer(config)

        with open(config_path, "w") as f:
            yaml.dump(config, f)
            
        with tarfile.open(fixed_nemo_path, "w") as tar:
            for root, dirs, files in os.walk(extract_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, extract_dir).replace("\\", "/")
                    tar.add(file_path, arcname=arcname)
                    
        shutil.rmtree(extract_dir)
        print(f"Successfully created patched model at: {fixed_nemo_path}")
    
    NEMO_MODEL_PATH = Path(fixed_nemo_path)

    print("Generating isolated Python 3.10 NeMo script...")
    nemo_script_path = str(WORK_DIR / "run_nemo_asr.py")
    with open(nemo_script_path, "w") as f:
        f.write(textwrap.dedent(f"""\
            import sys
            import csv
            import os
            import torch
            from pathlib import Path
            import nemo.collections.asr as nemo_asr

            device = "{device}"
            model_path = "{NEMO_MODEL_PATH}"
            csv_path = Path("{TRANSCRIPT_CSV}")
            batch_size = {ASR_BATCH_SIZE}
            audio_dir = Path("{FINAL_16K}")

            print("Loading NeMo ASR model:", model_path)
            # Override abstract method checks dynamically!
            # Since AI4Bharat didn't implement these, we mock them to bypass the TypeError.
            original_class = nemo_asr.models.ASRModel
            if not hasattr(original_class, "setup_training_data"):
                original_class.setup_training_data = lambda *args, **kwargs: None
            if not hasattr(original_class, "setup_validation_data"):
                original_class.setup_validation_data = lambda *args, **kwargs: None
                
            try:
                asr_model = original_class.restore_from(model_path, map_location=device, strict=False)
            except Exception as e:
                print(f"Error loading model: {{e}}")
                raise e
            
            asr_model.eval()
            asr_model = asr_model.to(device)

            existing = set()
            if csv_path.exists() and csv_path.stat().st_size > 0:
                with open(csv_path, "r", encoding="utf-8-sig") as h:
                    reader = csv.reader(h)
                    for row in reader:
                        if row and row[0] != "name":
                            existing.add(row[0])

            wavs = sorted(audio_dir.glob("*.wav"))
            pending = [str(w) for w in wavs if w.name not in existing]

            if not pending:
                print("No pending ASR tasks.")
                sys.exit(0)

            def normalize_asr_item(item) -> str:
                if isinstance(item, str):
                    return item.strip()
                if hasattr(item, "text"):
                    return str(item.text).strip()
                if isinstance(item, (list, tuple)) and item:
                    return normalize_asr_item(item[0])
                return str(item).strip()

            def batched(items, bsize):
                for start in range(0, len(items), bsize):
                    yield items[start:start + bsize]

            def transcribe_paths(paths):
                # Try permutations of NeMo API signatures
                for attempt in [
                    lambda: asr_model.transcribe(paths2audio_files=paths, batch_size=batch_size, return_hypotheses=False),
                    lambda: asr_model.transcribe(paths2audio_files=paths, batch_size=batch_size),
                    lambda: asr_model.transcribe(audio=paths, batch_size=batch_size),
                    lambda: asr_model.transcribe(paths, batch_size=batch_size),
                ]:
                    try:
                        output = attempt()
                        if isinstance(output, tuple): output = output[0]
                        return [normalize_asr_item(item) for item in output]
                    except TypeError:
                        continue
                raise ValueError("Could not find correct NeMo transcribe signature")

            print(f"Transcribing {{len(pending)}} chunks via NeMo...")
            with csv_path.open("a", encoding="utf-8", newline="") as h:
                writer = csv.writer(h)
                for batch in batched(pending, batch_size):
                    texts = transcribe_paths(batch)
                    for wav_path, text in zip(batch, texts):
                        text = text.strip()
                        if text:
                            writer.writerow([Path(wav_path).name, text])
                    h.flush()
            print("NeMo Transcriptions complete.")
        """))

    print("Running NeMo through isolated Python 3.10 environment...")
    run(["/opt/nemo_env/bin/python", nemo_script_path], check=True)

elif pending and ASR_ENGINE == "ai4bharat_transformers_fallback":
    import librosa
    import soundfile as sf
    import numpy as np
    import torch
    from transformers import AutoModel

    print(f"Using device: {device}")
    print("Loading AI4Bharat Conformer model...")
    model_name = "ai4bharat/indic-conformer-600m-multilingual"
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
    model.eval()

    def load_audio_for_asr(path, target_sr=16000):
        audio, sr = sf.read(str(path))
        if getattr(audio, "ndim", 1) > 1:
            audio = audio.mean(axis=1)
        if sr != target_sr:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        return audio.astype("float32")

    with TRANSCRIPT_CSV.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        
        for wav_path in tqdm(pending, desc="Transcribing"):
            try:
                audio = load_audio_for_asr(wav_path)
                audio_tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).to(device)
                
                with torch.no_grad():
                    transcription = model(audio_tensor, lang=LANGUAGE_ID)
                
                if isinstance(transcription, list):
                    transcription = transcription[0]
                    
                text = str(transcription).strip()
                if text:
                    writer.writerow([wav_path.name, text])
                handle.flush()
            except Exception as e:
                print(f"\nError transcribing {wav_path.name}: {e}")

print("ASR pass complete.")
print("Rows now:", len(read_existing_transcripts(TRANSCRIPT_CSV)))

## 9. Validate Dataset Quality

This checks coverage, sample rates, durations, clipping, blank text, and missing 22 kHz pairs.

In [ ]:
import numpy as np

transcripts = read_existing_transcripts(TRANSCRIPT_CSV)
audit_rows = []
total_duration = 0.0

for wav16 in sorted(FINAL_16K.glob("*.wav")):
    wav22 = FINAL_22K / wav16.name
    info16 = sf.info(str(wav16))
    audio, sr = sf.read(str(wav16), dtype="float32")
    if getattr(audio, "ndim", 1) > 1:
        audio = audio.mean(axis=1)
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    rms = float(np.sqrt(np.mean(np.square(audio)))) if audio.size else 0.0
    dur = float(info16.duration)
    total_duration += dur
    text = transcripts.get(wav16.name, "")
    issues = []
    if info16.samplerate != 16000:
        issues.append("bad_16k_samplerate")
    if not wav22.exists():
        issues.append("missing_22k_pair")
    else:
        info22 = sf.info(str(wav22))
        if info22.samplerate != 22050:
            issues.append("bad_22k_samplerate")
    if dur < 3.0:
        issues.append("very_short")
    if dur > 12.0:
        issues.append("very_long")
    if peak > 0.999:
        issues.append("possible_clipping")
    if rms < 1e-4:
        issues.append("near_silence")
    if not text:
        issues.append("missing_transcription")
    audit_rows.append({
        "name": wav16.name,
        "duration_sec": round(dur, 3),
        "sample_rate": info16.samplerate,
        "peak": round(peak, 6),
        "rms": round(rms, 6),
        "text_chars": len(text),
        "transcription": text,
        "issues": ";".join(issues),
    })

audit_df = pd.DataFrame(audit_rows)
audit_path = FINAL_ROOT / "dataset_audit.csv"
audit_df.to_csv(audit_path, index=False, encoding="utf-8")

missing_count = int(audit_df["issues"].str.contains("missing_transcription", regex=False).sum()) if len(audit_df) else 0
issue_count = int(audit_df["issues"].astype(bool).sum()) if len(audit_df) else 0

print("Chunks:", len(audit_df))
print("Duration:", f"{total_duration/60:.2f} minutes", f"({total_duration/3600:.2f} hours)")
print("Missing transcriptions:", missing_count)
print("Rows with any issue:", issue_count)
print("Audit CSV:", audit_path)

display(audit_df.head())
display(audit_df[audit_df["issues"].astype(bool)].head(20))
assert missing_count == 0, "Some audio chunks are missing transcription. Re-run the ASR cell."

## 10. Manual Listening Check

Before submission, listen to random samples and confirm:
- Text roughly matches speech.
- Audio is clean.
- No music-only/noise-only chunks.

In [ ]:
from IPython.display import Audio, display, Markdown

sample_df = audit_df.sample(min(10, len(audit_df)), random_state=42) if len(audit_df) else audit_df
for _, row in sample_df.iterrows():
    display(Markdown(f"**{row['name']}**  \n{row['transcription']}"))
    display(Audio(str(FINAL_16K / row["name"])))

## 11. Create Dataset Card and ZIP

Submit the ZIP file. It contains both 16 kHz and 22.05 kHz chunks.

In [ ]:
from datetime import datetime

card = f"""# Nepali TTS Dataset: {DATASET_ID}

Created: {datetime.now().isoformat(timespec='seconds')}

Source:
- YouTube URL: {YOUTUBE_URL}

Pipeline:
- yt-dlp audio download
- FullSubNet+ denoising
- Silero VAD speech splitting
- Speaker filtering: disabled
- ASR: {ASR_ENGINE}
- Language: {LANGUAGE_ID}

Contents:
- 16000Hz/{DATASET_ID}/: ASR-ready chunks + transcriptions.csv
- 22050Hz/{DATASET_ID}/: TTS training chunks + copied transcriptions.csv
- dataset_audit.csv
- dataset_card.md

Stats:
- Chunks: {len(audit_df)}
- Duration minutes: {total_duration/60:.2f}
- Duration hours: {total_duration/3600:.2f}
- Missing transcriptions: {missing_count}
- Rows with audit issues: {issue_count}

Notes:
- Please manually review random samples before accepting this dataset.
- Do not include copyrighted audio in public releases unless you have permission.
"""

(FINAL_ROOT / "dataset_card.md").write_text(card, encoding="utf-8")
shutil.copy2(TRANSCRIPT_CSV, FINAL_22K / "transcriptions.csv")

zip_path = WORK_DIR / f"{DATASET_ID}_tts_dataset.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in FINAL_ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(FINAL_ROOT))

print("ZIP ready:", zip_path)
print("ZIP size:", f"{zip_path.stat().st_size / 1024**2:.1f} MiB")

In [ ]:
# Download the final ZIP to your laptop.
from google.colab import files
files.download(str(zip_path))

## Optional: Save to Google Drive

Run this if browser download is unstable.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# drive_out = Path('/content/drive/MyDrive/nepali_tts_datasets')
# drive_out.mkdir(parents=True, exist_ok=True)
# shutil.copy2(zip_path, drive_out / zip_path.name)
# print('Saved to:', drive_out / zip_path.name)